# Predicting tomorrow's air quality in Ohio: the very simple version

**The idea:** look at **today** (air quality, weather, time of year) and guess **tomorrow's** air quality. Then check how good the guesses are on days the computer has never seen.

**How to run:** choose **Runtime > Run all**. When the first cell asks, upload the Excel file `AQI_Ohio_2022_2026_with_weather.xlsx`. Everything runs in less than a minute.

**Words you will see**
- **AQI** (Air Quality Index): higher means worse air. 0-50 is good, and above 100 is unhealthy for sensitive people.
- **Clue**: a fact we give the computer to help it guess, for example today's AQI.
- **MAE**: how many AQI points the guess is wrong by, on average. Smaller is better.
- **RMSE**: like MAE, but big mistakes count extra. Smaller is better.
- **Baseline**: the lazy guess "tomorrow will be the same as today". A model is only useful if it beats this.

**Note:** this is a shortened version of the poster analysis (12 clues and 3 models). Its numbers are close to the poster's, but not identical. The "tomorrow = today" row is exactly the same.

## Step 1. Get the data

Run the next cell. Click **Choose files** and pick the Excel file. Each row of the file is one county on one day: its AQI and the weather.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files

uploaded = files.upload()                             # a "Choose files" button appears: pick the Excel file
filename = list(uploaded)[0]                          # the name of the file you uploaded
df = pd.read_excel(filename)                          # read the Excel file into a table called df
df = df.rename(columns={"county Name": "County"})     # a shorter column name
df = df.sort_values(["County", "Date"]).reset_index(drop=True)   # one county after another, oldest day first

print(len(df), "rows |", df.County.nunique(), "counties |", df.Date.min().date(), "to", df.Date.max().date())
df[["Date", "County", "AQI"]].head()                  # the first 5 rows

## Step 2. Look at the data: when and where is the air worst?

First we put every day into a season (winter = Dec-Feb, spring = Mar-May, summer = Jun-Aug, fall = Sep-Nov). Then we average the AQI of each season.

In [ ]:
season_of_month = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring", 4: "Spring", 5: "Spring",
                   6: "Summer", 7: "Summer", 8: "Summer", 9: "Fall", 10: "Fall", 11: "Fall"}
df["Season"] = df.Date.dt.month.map(season_of_month)

by_season = df.groupby("Season").AQI.agg(["mean", "median"]).round(1)     # median = the middle value
by_season = by_season.loc[["Winter", "Spring", "Summer", "Fall"]]         # put the seasons in calendar order
print(by_season)

fig, ax = plt.subplots(figsize=(6, 3))
by_season["mean"].plot.bar(ax=ax, rot=0, title="Average AQI by season")
plt.show()

Now the same for counties. We leave out counties with less than one year of data, because their averages are not reliable.

In [ ]:
county_table = df.groupby("County").AQI.agg(["mean", "count"])      # count = number of days with data
county_table = county_table[county_table["count"] >= 365].sort_values("mean").round(1)

print("Cleanest counties:")
print(county_table.head(3))
print("\nMost polluted counties:")
print(county_table.tail(3))

**What this means**
- Summer is the worst season (average AQI about 52). The other seasons are about 41 to 44.
- The most polluted counties are Cuyahoga (Cleveland), Butler and Hamilton (Cincinnati area). The cleanest are Belmont and Athens.

## Step 3. Make the clues

We want to guess **tomorrow's AQI** in each county. So for every county and day we prepare the **answer** (`tomorrow`) and the **clues** the computer may use:

| Clue | What it is |
|---|---|
| `aqi_today` | today's AQI in the county |
| `aqi_yesterday` | yesterday's AQI |
| `aqi_7_day_average` | the average AQI of the last 7 days |
| `month` | the month number (1 = January ... 12 = December) |
| `county_average` | the county's usual AQI in 2022-2024 |
| 7 weather columns | today's temperature, rain, wind, humidity and sunshine |

In [ ]:
each_county = df.groupby("County")                    # every calculation below stays inside one county

df["tomorrow"] = each_county.AQI.shift(-1)            # the ANSWER: the AQI on the next row
df["aqi_today"] = df.AQI
df["aqi_yesterday"] = each_county.AQI.shift(1)        # the AQI on the row before
df["aqi_7_day_average"] = each_county.AQI.transform(lambda day: day.rolling(7).mean())   # today and the 6 days before
df["month"] = df.Date.dt.month

# The county's usual AQI. We use 2022-2024 only, so the test days (2025 and later) stay unseen.
usual = df[df.Date < "2025-01-01"].groupby("County").AQI.mean()
df["county_average"] = df.County.map(usual)

In [ ]:
weather = ["Temp Max (°F)", "Temp Mean (°F)", "Precipitation (in)", "Wind Speed Max (mph)",
           "Wind Gust Max (mph)", "Relative Humidity (%)", "Solar Radiation (MJ/m²)"]
clues = ["aqi_today", "aqi_yesterday", "aqi_7_day_average", "month", "county_average"] + weather

# Keep only the days that have every clue, and where the next row really is the next day
next_row_is_tomorrow = (each_county.Date.shift(-1) - df.Date).dt.days == 1
df = df[next_row_is_tomorrow].dropna(subset=clues + ["tomorrow"])
print("Days we can use:", len(df))

## Step 4. Learn from the past, test on the future

We **train** (let the computer learn) on the days before 2025, and we **test** (check its guesses) on 2025 and later. We never mix them, because a real forecast cannot see the future.

In [ ]:
train = df[df.Date < "2025-01-01"]        # 2022-2024: the computer learns from these days
test = df[df.Date >= "2025-01-01"]        # 2025 to May 2026: we check its guesses on these days

X_train, y_train = train[clues], train.tomorrow     # X = the clues, y = the answers
X_test, y_test = test[clues], test.tomorrow
print("Training days:", len(train), "| Test days:", len(test))

## Step 5. Train the models and compare them

We compare three ways of guessing:
- **Baseline:** tomorrow = today. No learning at all.
- **Linear regression:** a straight-line formula that mixes the clues.
- **Random Forest:** 100 decision trees that each make a guess, then vote. Many simple models working together is called an *ensemble*.

The poster also tests Ridge, Gradient Boosting and a tuned forest. They land close to these results, so we keep only three models here.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

line = LinearRegression()
line.fit(X_train, y_train)                         # learn the straight-line formula from the training days

forest = RandomForestRegressor(n_estimators=100,       # 100 trees
                               min_samples_leaf=10,    # every end-group of a tree must hold at least 10 days
                               max_features=0.5,       # each split looks at a random half of the clues
                               random_state=42)        # a fixed number, so you get the same result every run
forest.fit(X_train, y_train)                       # grow the trees from the training days
print("Both models have learned from", len(X_train), "training days.")

In [ ]:
# Every model guesses tomorrow's AQI for all the test days
guesses = {"Baseline": X_test.aqi_today,           # the lazy guess: tomorrow = today
           "Linear regression": line.predict(X_test),
           "Random Forest": forest.predict(X_test)}

# Grade each guess against the real AQI
errors = {}
for name, guess in guesses.items():
    errors[name] = {"MAE": mean_absolute_error(y_test, guess),
                    "RMSE": mean_squared_error(y_test, guess) ** 0.5}

table = pd.DataFrame(errors).T                     # one row per model
print(table.round(2))

In [ ]:
better = 1 - table.RMSE["Random Forest"] / table.RMSE["Baseline"]
print(f"The Random Forest's RMSE is {better:.0%} lower than the baseline.")

fig, ax = plt.subplots(figsize=(7, 2.5))
table.MAE.plot.barh(ax=ax, title="Average miss in AQI points (smaller is better)")
ax.invert_yaxis()                                  # list the models top to bottom, as in the table
plt.show()

**What this means**
- Both real models beat the lazy baseline by a clear margin: their average miss is about 8.6 to 8.8 AQI points instead of 10.3.
- The straight-line formula already gets most of the improvement. The forest adds a little more and is the best of the three.

## Step 6. What does the forest rely on?

While it learns, the forest keeps track of which clues it used the most. A longer bar means a more important clue.

In [ ]:
importance = pd.Series(forest.feature_importances_, index=clues).sort_values()    # smallest first

fig, ax = plt.subplots(figsize=(7, 4))
importance.plot.barh(ax=ax, title="Which clues does the forest use the most?")
plt.show()

**What this means**
- Today's AQI is by far the most important clue.
- The 7-day average and sunshine come next. The county, wind and the other clues matter a little each.

## Step 7. Where does it fail?

The very bad days matter most for health. Let's see how the forest did on the test days when tomorrow's AQI was above 100.

In [ ]:
guess = pd.Series(forest.predict(X_test), index=y_test.index)      # the forest's guess for every test day

bad_days = y_test > 100                                            # days when tomorrow's AQI was above 100
print("Test days with AQI above 100:", bad_days.sum())
print("Of those, days where the forest guessed above 100:", (guess[bad_days] > 100).sum())

In [ ]:
# One county, summer 2025: the real AQI against the forest's guess
county = county_table.index[-1]                                    # the most polluted county
summer = test[(test.County == county) & (test.Date >= "2025-06-01") & (test.Date < "2025-09-01")]
guess_for = summer.Date + pd.Timedelta(days=1)                     # a guess made today is for tomorrow

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(guess_for, summer.tomorrow, label="Real AQI")
ax.plot(guess_for, guess[summer.index], label="Forest's guess")
ax.set_title(f"{county} County, summer 2025")
ax.legend()
plt.show()

**What this means**
- The forest follows the normal ups and downs, but it does not see the spikes coming: it did not guess "above 100" for any of the bad days.
- So it is useful on normal days, but it is **not a warning system** for the worst days.

## Summary in plain English

- **Where and when:** the air is worst in summer and around the big cities (Cleveland and Cincinnati).
- **Guessing tomorrow:** the forest is off by about 8.6 AQI points on average. The lazy "same as today" guess is off by 10.3. In RMSE terms that is about 17% better.
- **What matters most:** today's AQI, then the 7-day average and sunshine. The county and the other weather add a little.
- **Limit:** it works on normal days but misses the spikes, so it is not a warning system yet.
- **Next step:** add next-day weather forecasts and wildfire smoke data to help with the spikes.